# 06 — CF-Miner: Úloha 2
## Profilování dle věku a povrchu
### 4IZ503 Projektový seminář — Ultra Marathon Running

---

### Slovní zadání

Existují kombinace věkové skupiny + povrchu (+ pohlaví), kde je profil
`speed_cat` výrazně odlišný od celkového průměru?

**Hypotéza:** Starší závodníci (50+) mají výhodu na trail závodech
oproti silničním, protože trail penalizuje rychlost ve prospěch
vytrvalosti a zkušenosti s terénem. Profil speed_cat pro
`age_group(50-59) ∧ surface(trail)` bude vyrovnanější než pro
`age_group(18-29) ∧ surface(trail)`.

**Target:** `speed_cat` (3 kategorie: pomalý / střední / rychlý)  
**Podmínky:** kombinace `age_group`, `surface`, `gender`

> ⚠️ **Poznámka k datům:** `surface` má ~84 % null — dataset filtrován
> pouze na záznamy se známým povrchem (33 závodů s metadaty).

---

### Parametry úlohy

| Parametr | Hodnota |
|---|---|
| Procedura | CF-Miner |
| Target | speed_cat |
| Base (min. počet záznamů) | 500 |
| RelMax_leq (max. relativní podíl dominantní kategorie) | 0.45 |
| Cond atributy | age_group, surface, gender (maxlen=3) |
| Data | ultra_clean_cm.parquet, filtrováno na surface notna() |

## 1. Import a načtení dat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from cleverminer import cleverminer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')

df_cm = pd.read_parquet(DATA_DIR / 'ultra_clean_cm.parquet')
print(f"Načteno: {len(df_cm):,} řádků")
print(f"Sloupce: {df_cm.columns.tolist()}")

## 2. Příprava dat pro úlohu

In [ ]:
# Filtrujeme na záznamy se známým povrchem (84% null — jen 33 závodů)
cols = ['age_group', 'surface', 'gender', 'speed_cat']
df_task = df_cm[cols].dropna().copy()

print(f"Záznamy se známým povrchem: {len(df_task):,}")
print(f"(z celkových {len(df_cm):,} = {len(df_task)/len(df_cm)*100:.1f} %)")
print()
print("Rozložení age_group:")
print(df_task['age_group'].value_counts().sort_index())
print()
print("Rozložení surface:")
print(df_task['surface'].value_counts())
print()
print("Rozložení gender:")
print(df_task['gender'].value_counts())
print()
print("Rozložení speed_cat (ověření ~33/33/33):")
print((df_task['speed_cat'].value_counts() / len(df_task) * 100).round(1))

## 3. CleverMiner — CF-Miner úloha

In [ ]:
cm = cleverminer(df=df_task)

cm.mine(
    proc='CFMiner',
    target='speed_cat',
    quantifiers={'Base': 500, 'RelMax_leq': 0.45},
    cond={
        'attributes': [
            {'name': 'age_group', 'type': 'subset', 'minlen': 1, 'maxlen': 1},
            {'name': 'surface',   'type': 'subset', 'minlen': 1, 'maxlen': 1},
            {'name': 'gender',    'type': 'subset', 'minlen': 1, 'maxlen': 1},
        ],
        'minlen': 1, 'maxlen': 3, 'type': 'con'
    }
)

print("\nSouhrn:")
cm.print_summary()

## 4. Výsledky

In [ ]:
print("Všechna pravidla (seřazená dle RelMax):")
cm.print_rulelist(sortby='relmax', storesorted=True)

## 5. Extrakce pravidel pro analýzu

In [ ]:
rules = []
n = cm.get_rulecount()

for i in range(1, n + 1):
    quant = cm.get_quantifiers(i)
    rule_text = cm.get_ruletext(i)

    # Parsování podmínek z textu pravidla
    age_match     = re.search(r'age_group\(([^)]+)\)', rule_text)
    surface_match = re.search(r'surface\((\w+)\)', rule_text)
    gender_match  = re.search(r'gender\((\w+)\)', rule_text)

    rules.append({
        'rule_id':   i,
        'age_group': age_match.group(1)     if age_match     else None,
        'surface':   surface_match.group(1) if surface_match else None,
        'gender':    gender_match.group(1)  if gender_match  else None,
        'base':      quant.get('base'),
        'relmax':    quant.get('relmax'),
        'rule_text': rule_text,
    })

df_rules = pd.DataFrame(rules)
print(f"Extrahováno {len(df_rules)} pravidel")
print()
print(df_rules.sort_values('relmax').to_string(index=False))

## 6. Vizualizace

In [ ]:
age_order    = ['18-29', '30-39', '40-49', '50-59', '60-69', '70+']
surface_colors = {'road': '#5b9bd5', 'trail': '#70ad47', 'mixed': '#f4b942'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('CF-Miner: Profilování dle věku a povrchu\n'
             '(target: speed_cat, RelMax ≤ 0.45, surface notna)',
             fontsize=13, fontweight='bold')

# Graf 1 — Počet pravidel dle surface
surf_counts = df_rules['surface'].value_counts()
surfs = [s for s in ['road', 'trail', 'mixed'] if s in surf_counts.index]
bars = ax1.bar(surfs,
               [surf_counts.get(s, 0) for s in surfs],
               color=[surface_colors[s] for s in surfs],
               alpha=0.85, edgecolor='white')
for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 0.1,
             str(int(h)), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_xlabel('Povrch', fontsize=11)
ax1.set_ylabel('Počet splněných pravidel', fontsize=11)
ax1.set_title('Počet CF-Miner pravidel dle povrchu', fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# Graf 2 — RelMax dle věkové skupiny (průměr přes všechny povrchy)
df_age = df_rules[df_rules['age_group'].notna()].copy()
if len(df_age) > 0:
    relmax_by_age = df_age.groupby('age_group')['relmax'].mean().reindex(age_order)
    age_cats = [a for a in age_order if a in relmax_by_age.index and not pd.isna(relmax_by_age[a])]
    x = np.arange(len(age_cats))

    bars2 = ax2.bar(x, [relmax_by_age[a] for a in age_cats],
                    color='#9b59b6', alpha=0.85, edgecolor='white')
    for bar in bars2:
        h = bar.get_height()
        if h > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    ax2.axhline(0.333, color='black', linewidth=1, linestyle='--', label='Rovnoměrné rozložení (33%)')
    ax2.axhline(0.45,  color='red',   linewidth=1, linestyle=':', label='Práh RelMax_leq=0.45')
    ax2.set_xlabel('Věková skupina', fontsize=11)
    ax2.set_ylabel('Průměrné RelMax', fontsize=11)
    ax2.set_title('Průměrný RelMax dle věkové skupiny\n'
                  '(nižší = vyrovnanější profil speed_cat)', fontsize=11)
    ax2.set_xticks(x)
    ax2.set_xticklabels(age_cats, fontsize=9)
    ax2.legend(fontsize=9)
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim(0.25, 0.50)
else:
    ax2.text(0.5, 0.5, 'Žádná pravidla s age_group', ha='center', va='center',
             transform=ax2.transAxes, fontsize=12)
    ax2.set_title('RelMax dle věkové skupiny', fontsize=11)

plt.tight_layout()
plt.savefig(DATA_DIR / '06_vek_povrch.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graf uložen.")

## 7. Interpretace grafů

**Graf vlevo — Počet pravidel dle povrchu:**

Povrch s více pravidly CF-Mineru má více kombinací podmínek vedoucích
k diverzifikovanému profilu výkonnosti. Trail závody typicky
favorizují zkušenost — menší dominance rychlé skupiny.

**Graf vpravo — Průměrný RelMax dle věkové skupiny:**

Věkové skupiny s nižším průměrným RelMax mají vyrovnanější
zastoupení rychlých/středních/pomalých závodníků. Starší věkové
skupiny (50+) mohou mít odlišný profil díky selekčnímu efektu
(jen zkušenější závodníci startují ultra ve vyšším věku).

## 8. Zajímavá pravidla

In [ ]:
print("=== ZAJÍMAVÁ PRAVIDLA ===")
print()

df_sorted = df_rules.sort_values('relmax')
print("TOP 3 pravidla (nejnižší RelMax — nejrovnoměrnější profil speed_cat):")
for _, row in df_sorted.head(3).iterrows():
    cm.print_rule(int(row['rule_id']))
    print()

## 9. Souhrn a business interpretace

In [ ]:
print("=" * 60)
print("SOUHRN — Úloha 2 (CF): Profilování dle věku a povrchu")
print("=" * 60)
print()

print(f"Celkem nalezených pravidel: {len(df_rules)}")
print(f"Záznamy použité pro mining: {len(df_task):,} (surface notna)")
print()

# Srovnání dle povrchu
print("Pravidla dle povrchu:")
for surf in ['road', 'trail', 'mixed']:
    cnt = len(df_rules[df_rules['surface'] == surf])
    print(f"  {surf:6s}: {cnt} pravidel")
print()

if len(df_rules) > 0:
    best = df_rules.sort_values('relmax').iloc[0]
    print(f"Nejrovnoměrnější profil: {best['rule_text']}")
    print(f"  RelMax = {best['relmax']:.3f}, Base = {best['base']:,.0f}")

print()
print("BUSINESS DOPORUČENÍ:")
print("  → Trail závody pro starší věkové skupiny mohou být lépe dostupné")
print("  → Organizátoři trail závodů: cílit marketing na závodníky 40-59 let")
print("  → Road závody s homogenním věkovým profilem favorizují mladší závodníky")

## Shrnutí

**Metoda:** CF-Miner (CleverMiner 1.2.6). Podmínky tvořeny kombinacemi
`age_group`, `surface`, `gender`. Target: `speed_cat`.
Kvantifikátory: Base ≥ 500, RelMax ≤ 0.45.

**Data:** Filtrováno na záznamy se známým povrchem (~16 % datasetu,
odpovídá 33 závodům s metadaty).

**Klíčový nález:** Povrch a věk ovlivňují vyrovnanost výkonnostního
profilu pole — trail závody mohou mít diverzifikovanější zastoupení
starších závodníků.

**Limitace:**
- 84 % null v surface — výsledky platí jen pro 33 závodů se známými metadaty
- Malá subpopulace (70+) může nedosahovat minimálního Base
- surface a age_group korelují s geografií závodu

**Další notebook:** `07_SD4ft_uloha1.ipynb` — Ženy vs. muži dle věkové skupiny (SD4ft-Miner)